<a href="https://colab.research.google.com/github/tanphong-sudo/voxbridge/blob/main/colab/00_dev_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VoxBridge - Colab Development Bootstrap

Use this notebook for ordinary source-code development. It clones or fast-forward updates the
private repository, restores the validated Drive model cache into `/content/voxbridge/models`, and
runs the real JVM E2E gate.

Do **not** run model export here. Add a `GITHUB_TOKEN` with private-repository `repo` access to
Colab Secrets; the token is read at runtime and is never written to the notebook or Git checkout.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/VoxBridge"


## 2. Clone or update the private source repository

In [ ]:
import os
import pathlib
import subprocess
from google.colab import userdata

REPO_URL = "https://github.com/tanphong-sudo/voxbridge.git"
REPO = pathlib.Path("/content/voxbridge")


def run_git(args, cwd=None):
    """Run Git with the token held only in this process environment."""
    token = userdata.get("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Missing GITHUB_TOKEN. In the left sidebar, open Secrets, add GITHUB_TOKEN, "
            "and enable its notebook access. The token needs read access to voxbridge."
        )

    askpass = pathlib.Path("/tmp/voxbridge-git-askpass.sh")
    askpass.write_text(
        "#!/bin/sh\n"
        "case \"$1\" in\n"
        "  *Username*) echo x-access-token ;;\n"
        "  *) printf '%s\\n' \"$GITHUB_TOKEN\" ;;\n"
        "esac\n"
    )
    askpass.chmod(0o700)
    env = {
        **os.environ,
        "GITHUB_TOKEN": token,
        "GIT_ASKPASS": str(askpass),
        "GIT_TERMINAL_PROMPT": "0",
    }
    try:
        return subprocess.run(["git", *args], cwd=cwd, env=env, text=True, check=True)
    finally:
        askpass.unlink(missing_ok=True)


def sync_source():
    if not (REPO / ".git").is_dir():
        if REPO.exists():
            raise RuntimeError(
                f"{REPO} exists but is not a Git checkout. Run: !rm -rf /content/voxbridge, then rerun this cell."
            )
        run_git(["clone", "--branch", "main", REPO_URL, str(REPO)])
    else:
        run_git(["fetch", "origin", "main"], cwd=REPO)
        run_git(["pull", "--ff-only", "origin", "main"], cwd=REPO)

    commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True
    ).strip()
    print(f"Source ready: {REPO} @ {commit}")


sync_source()


## 3. Restore the validated model cache

In [ ]:
import subprocess

restore = subprocess.run(
    ["bash", "tools/dev/restore-models.sh", f"{DRIVE_ROOT}/models", "models"],
    cwd=REPO,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(restore.stdout)
if restore.returncode:
    raise RuntimeError(
        "Model restore failed. The lines above identify the missing Drive directory or model artifacts. "
        "Do not run E2E until this cell succeeds."
    )


## 4. Run real JVM E2E

In [ ]:
import subprocess
subprocess.run(["bash", "tools/dev/status.sh", "models"], cwd=REPO, check=True)
subprocess.run(["bash", "tools/e2e-jvm/run.sh", "models"], cwd=REPO, check=True)
print("\n## VoxBridge Dev Environment")
print("Git commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip())
print("Models cache: OK")
print("E2E: completed; inspect PASS/FAIL output above.")


## UPDATE SOURCE + TEST

Run this cell after every pushed Kotlin/runtime/test change. It does not restore, replace, or export models.


In [ ]:
# UPDATE SOURCE + TEST - normal loop after a pushed source-code change.
# Run the source-sync cell once after opening this notebook so run_git() and sync_source() exist.
sync_source()

subprocess.run(["bash", "tools/dev/status.sh", "models"], cwd=REPO, check=True)
subprocess.run(["bash", "tools/e2e-jvm/run.sh", "models"], cwd=REPO, check=True)
print("\n## VoxBridge Dev Environment")
print("Git commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip())
print("Models cache: OK")
print("E2E: completed; inspect PASS/FAIL output above.")
